<a href="https://colab.research.google.com/github/Ajmal30/Vllm/blob/main/VLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    print("No GPU detected")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [3]:
!nvidia-smi

Sat Aug 29 15:12:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!pip install -q transformers accelerate sentencepiece

In [5]:
import transformers
import accelerate

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)

Transformers: 5.15.1
Accelerate: 1.14.0


## Choosing a Small Open-Weight Causal Language Model for Inference Engineering

Given your setup with a Tesla T4 GPU (approximately 15 GB VRAM), the goal is to select a model that balances learning value with practical VRAM constraints. Here's a comparison of 2-3 suitable options:

### Model Comparison:

1.  **TinyLlama/TinyLlama-1.1B-Chat-v1.0**
    *   **Parameter Count:** 1.1 Billion parameters
    *   **VRAM Requirements:** Approximately 2.2 GB (float16) to 4.4 GB (float32) for model weights alone. Inference with batching and KV cache will add to this, but it's well within your 15 GB VRAM.
    *   **Architecture:** Llama-like architecture, which is widely adopted and provides excellent learning opportunities due to its popularity and optimizations.
    *   **Learning Value:** Excellent. It's small enough for rapid experimentation, but complex enough to demonstrate key inference engineering concepts like quantization, batching, and KV caching. Being a chat model, it also offers practical experience with conversational AI.

2.  **facebook/opt-1.3b**
    *   **Parameter Count:** 1.3 Billion parameters
    *   **VRAM Requirements:** Similar to TinyLlama-1.1B, roughly 2.6 GB (float16) to 5.2 GB (float32) for model weights.
    *   **Architecture:** Transformer-based, developed by Meta. It's a foundational model that has influenced many subsequent architectures.
    *   **Learning Value:** High. It's a good reference point for understanding the core Transformer architecture without the complexity of larger models. It's a good alternative to Llama-like models for architectural comparison.

3.  **google/gemma-2b-it**
    *   **Parameter Count:** 2 Billion parameters
    *   **VRAM Requirements:** Approximately 4 GB (float16) to 8 GB (float32) for model weights. Still comfortably within your T4's VRAM.
    *   **Architecture:** Developed by Google, inspired by the Gemini models. It uses a modern Transformer architecture.
    *   **Learning Value:** Very high. Gemma models are state-of-the-art for their size and provide insights into Google's approach to efficient model design. The instruction-tuned (`-it`) version is excellent for learning prompt engineering alongside inference engineering.

### Recommendation:

I recommend starting with **google/gemma-2b-it**.

**Why:**

*   **Modern Architecture:** It represents a cutting-edge design from Google, offering insights into contemporary LLM development and optimizations.
*   **Instruction-Tuned:** The `-it` variant is instruction-tuned, making it particularly useful for learning prompt engineering techniques, which are crucial for effective inference. This adds an extra layer of practical learning beyond just the technical aspects of inference.
*   **Optimal Size:** At 2 billion parameters, it's large enough to exhibit significant LLM behaviors and challenges (like memory management and latency) but still small enough to fit comfortably on your Tesla T4, allowing for experimentation with various optimizations (quantization, different precision types, etc.) without hitting VRAM limits too quickly.
*   **Strong Performance:** For its size, Gemma-2B-it generally performs very well, which means your inference results will be more meaningful and less prone to

### Loading the Model and Tokenizer

We'll use the `AutoTokenizer` and `AutoModelForCausalLM` classes from the `transformers` library to load the `TinyLlama/TinyLlama-1.1B-Chat-v1.0` model. We'll specify `torch.bfloat16` as the data type for efficiency, as the Tesla T4 supports bfloat16. The model will be moved to the GPU (`cuda`) as it loads.

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from huggingface_hub import login

# Changed model to a publicly available one for ease of access and learning
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Login to Hugging Face Hub if you need to access other gated models in the future.
# For TinyLlama, it's not strictly necessary.
# login()

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model and move to GPU
# Use torch.bfloat16 for memory efficiency on Tesla T4, if available and supported.
# Otherwise, you might use torch.float16 or default float32.
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print(f"Model '{model_name}' loaded successfully!")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model 'TinyLlama/TinyLlama-1.1B-Chat-v1.0' loaded successfully!


### Inspecting Model Properties

Now, let's examine various properties of the loaded model, including its parameter count, data type, device, configuration, and the GPU memory it consumes.

In [7]:
print("\n--- Model Inspection ---")

# 1. Parameter Count
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Model Parameters: {total_params:,}")

# 2. Data Type (dtype)
print(f"Model Data Type (dtype): {model.dtype}")

# 3. Device
# Check the device of the first parameter to infer the model's device
model_device = next(model.parameters()).device
print(f"Model Device: {model_device}")

# 4. Configuration
print("\nModel Configuration:")
print(model.config)

# 5. GPU Memory Usage
if torch.cuda.is_available():
    allocated_memory = torch.cuda.memory_allocated() / (1024**3)
    max_allocated_memory = torch.cuda.max_memory_allocated() / (1024**3)
    print(f"\nGPU Memory Allocated: {allocated_memory:.2f} GB")
    print(f"GPU Max Memory Allocated (since start): {max_allocated_memory:.2f} GB")
else:
    print("\nGPU not available, cannot show GPU memory usage.")


--- Model Inspection ---
Total Model Parameters: 1,100,048,384
Model Data Type (dtype): torch.bfloat16
Model Device: cuda:0

Model Configuration:
LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "dtype": "bfloat16",
  "eos_token_id": 2,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 5632,
  "max_position_embeddings": 2048,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 22,
  "num_key_value_heads": 4,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "rope_theta": 10000.0,
    "rope_type": "default"
  },
  "tie_word_embeddings": false,
  "transformers_version": "5.15.1",
  "use_cache": true,
  "vocab_size": 32000
}


GPU Memory Allocated: 2.05 GB
GPU Max Memory Allocated (since start): 2.05 GB


### Inspecting Input Tensor Shapes

To understand the tensor shapes, let's tokenize a simple prompt and then examine the `input_ids` and `attention_mask`.

In [8]:
# Define a simple prompt
prompt = "Hello, how are you today?"

# Tokenize the prompt
# return_tensors="pt" ensures PyTorch tensors are returned
inputs = tokenizer(prompt, return_tensors="pt")

# Move inputs to the same device as the model
inputs = {k: v.to(model.device) for k, v in inputs.items()}

print(f"Prompt: '{prompt}'")
print(f"Input IDs shape: {inputs['input_ids'].shape}")
print(f"Attention Mask shape: {inputs['attention_mask'].shape}")

Prompt: 'Hello, how are you today?'
Input IDs shape: torch.Size([1, 8])
Attention Mask shape: torch.Size([1, 8])


### Understanding B, S, and H in Transformer Inference

In transformer models, the typical dimensions you'll encounter for input tensors are represented by `(B, S)` or `(B, S, H)`:

*   **B (Batch Size):** This is the number of sequences (prompts or parts of prompts) that are processed simultaneously by the model. When you send multiple prompts or a single prompt split into multiple chunks, `B` will be greater than 1. In our example above, `B=1` because we're processing a single prompt.

*   **S (Sequence Length):** This represents the number of tokens in each sequence within the batch. For `input_ids` and `attention_mask`, `S` corresponds to the tokenized length of the prompt. Different prompts can have different sequence lengths, but for batching, they are often padded to a uniform maximum sequence length within that batch.

*   **H (Hidden Size / Embedding Dimension):** This dimension appears in intermediate representations within the model, such as after the token embeddings or in the outputs of attention layers. It represents the dimensionality of the vector space where each token or position is represented. For the raw `input_ids` or `attention_mask`, `H` is not present, as they are simply token indices or binary flags. However, if you were to look at the token embeddings (`model.get_input_embeddings()(input_ids)`), their shape would be `(B, S, H)`.

## Inside a Causal Transformer: A Single Inference Forward Pass (TinyLlama-1.1B)

Let's trace the data flow through our `TinyLlama/TinyLlama-1.1B-Chat-v1.0` model for a single inference forward pass, focusing on the tensor dimensions. For our example, we'll assume an input of `(B, S)` where `B=1` (batch size) and `S=8` (sequence length), as observed in our previous step.

From our model's configuration (`model.config`), we know:
*   `hidden_size = 2048`
*   `num_attention_heads = 32`
*   `num_key_value_heads = 4` (This is for Grouped Query Attention, a common optimization in Llama/Gemma. It means K and V share groups of heads, reducing computation and memory for K/V projections.)
*   `head_dim = 64`
*   `vocab_size = 32000`
*   `num_hidden_layers = 22`

### 1. Input Token IDs to Embeddings

*   **Input:** `input_ids` tensor, representing token indices.
    *   **Shape:** `(B, S)` -> `(1, 8)`
*   **Process:** The `input_ids` are looked up in the model's token embedding table.
*   **Output:** Token embeddings.
    *   **Shape:** `(B, S, hidden_size)` -> `(1, 8, 2048)`

### 2. Positional Information (RoPE - Rotary Positional Embeddings)

*   **Input:** The token embeddings (conceptually, RoPE applies to Q and K queries/keys).
    *   **Shape:** `(B, S, hidden_size)`
*   **Process:** RoPE is applied to the query and key vectors within each attention head. Unlike traditional positional embeddings that are *added* to token embeddings, RoPE applies a rotation matrix based on token position, encoding relative positional information directly into the attention mechanism. This happens *after* the linear projections of Q and K, but *before* the dot-product attention.
*   **Output:** The embeddings are implicitly modified as Q and K are rotated. The overall shape of the embeddings themselves remains `(B, S, hidden_size)` as they pass into the Transformer blocks.

### 3. Transformer Block (Repeated `num_hidden_layers` times, i.e., 22 times)

Each transformer block (or layer) consists of two main sub-layers: a Multi-Head Self-Attention mechanism and a Feed-Forward Network (MLP), with residual connections and layer normalization around each.

Let `X` be the input to a transformer block: `(B, S, hidden_size)` -> `(1, 8, 2048)`

#### a. Layer Normalization (Pre-Normalization)

*   **Input:** `X`
*   **Process:** Apply RMSNorm (a type of layer normalization used in Llama/Gemma) to `X`. This normalizes the feature dimension (`hidden_size`).
*   **Output:** `X_norm`
    *   **Shape:** `(B, S, hidden_size)` -> `(1, 8, 2048)`

#### b. Multi-Head Self-Attention (MHA)

*   **Input:** `X_norm`
    *   **Shape:** `(B, S, hidden_size)` -> `(1, 8, 2048)`
*   **Process:**
    1.  **Linear Projections:** `X_norm` is projected into Query (`Q`), Key (`K`), and Value (`V`) vectors. With Grouped Query Attention (`num_key_value_heads = 4`), Q will have `num_attention_heads` (32) heads, while K and V will have `num_key_value_heads` (4) heads. Each head gets `head_dim` (64) features.
        *   `Q_proj`: `X_norm` -> `(B, S, num_attention_heads * head_dim)` -> `(1, 8, 32 * 64)` -> `(1, 8, 2048)`
        *   `K_proj`: `X_norm` -> `(B, S, num_key_value_heads * head_dim)` -> `(1, 8, 4 * 64)` -> `(1, 8, 256)`
        *   `V_proj`: `X_norm` -> `(B, S, num_key_value_heads * head_dim)` -> `(1, 8, 4 * 64)` -> `(1, 8, 256)`
    2.  **Reshape for Attention Heads:** Q, K, V are reshaped to separate the heads.
        *   `Q`: `(B, num_attention_heads, S, head_dim)` -> `(1, 32, 8, 64)`
        *   `K`: `(B, num_key_value_heads, S, head_dim)` -> `(1, 4, 8, 64)`
        *   `V`: `(B, num_key_value_heads, S, head_dim)` -> `(1, 4, 8, 64)`
    3.  **RoPE Application:** Rotary positional embeddings are applied to `Q` and `K` at this stage, effectively rotating their vectors based on position. The shape remains the same.
    4.  **Key-Value Cache:** For efficient generation (inference), the `K` and `V` values from previous tokens are typically *cached*. For a new token, the current `K` and `V` are appended to the cache.
    5.  **Repeat K/V Heads:** Since `num_attention_heads > num_key_value_heads`, the K and V tensors are repeated (or simply broadcasted during computation) along the head dimension so that each query head can attend to all K/V information. This changes their effective shape for the dot product to `(B, num_attention_heads, S, head_dim)`.
    6.  **Scaled Dot-Product Attention:**
        *   Calculate attention scores: `Q @ K^T / sqrt(head_dim)` -> `(B, num_attention_heads, S, S)` -> `(1, 32, 8, 8)`
        *   Apply **causal mask**: For a causal model, each token can only attend to previous tokens and itself. This mask sets future token scores to negative infinity (`-inf`) before softmax.
        *   Apply Softmax: Normalizes scores to sum to 1.
        *   Multiply by `V`: `Attention_Scores @ V` -> `(B, num_attention_heads, S, head_dim)` -> `(1, 32, 8, 64)`
    7.  **Concatenate Heads:** The output from all attention heads is concatenated back along the `hidden_size` dimension.
        *   **Shape:** `(B, S, num_attention_heads * head_dim)` -> `(1, 8, 32 * 64)` -> `(1, 8, 2048)`
    8.  **Output Linear Projection:** The concatenated output is passed through a final linear layer.
        *   **Shape:** `(B, S, hidden_size)` -> `(1, 8, 2048)`
*   **Output:** `Attention_Output`
    *   **Shape:** `(B, S, hidden_size)` -> `(1, 8, 2048)`

#### c. Residual Connection 1

*   **Input:** `X` (original input to the block) and `Attention_Output`.
*   **Process:** Add `Attention_Output` to `X` (`X + Attention_Output`). This allows gradients to flow directly through the network.
*   **Output:** `Residual_1_Output`
    *   **Shape:** `(B, S, hidden_size)` -> `(1, 8, 2048)`

#### d. Layer Normalization (Pre-Normalization)

*   **Input:** `Residual_1_Output`
*   **Process:** Apply RMSNorm.
*   **Output:** `Residual_1_Output_norm`
    *   **Shape:** `(B, S, hidden_size)` -> `(1, 8, 2048)`

#### e. Feed-Forward Network (MLP)

*   **Input:** `Residual_1_Output_norm`
    *   **Shape:** `(B, S, hidden_size)` -> `(1, 8, 2048)`
*   **Process:** This consists of several linear layers with activation functions (e.g., GELU or SiLU as in Gemma/Llama) in between. Typically, the input `hidden_size` is expanded to an `intermediate_size` and then projected back to `hidden_size`. For TinyLlama, `intermediate_size` is `5632`.
    1.  Linear layer 1: `(B, S, hidden_size)` -> `(B, S, intermediate_size)` -> `(1, 8, 5632)`
    2.  Activation Function (e.g., SiLU)
    3.  Linear layer 2: `(B, S, intermediate_size)` -> `(B, S, hidden_size)` -> `(1, 8, 2048)`
*   **Output:** `MLP_Output`
    *   **Shape:** `(B, S, hidden_size)` -> `(1, 8, 2048)`

#### f. Residual Connection 2

*   **Input:** `Residual_1_Output` and `MLP_Output`.
*   **Process:** Add `MLP_Output` to `Residual_1_Output` (`Residual_1_Output + MLP_Output`).
*   **Output:** The final output of one transformer block.
    *   **Shape:** `(B, S, hidden_size)` -> `(1, 8, 2048)`

This entire Transformer Block process is repeated `num_hidden_layers` (22) times.

### 4. Final Layer Normalization

*   **Input:** The output from the last transformer block.
    *   **Shape:** `(B, S, hidden_size)` -> `(1, 8, 2048)`
*   **Process:** Apply a final RMSNorm.
*   **Output:** `Normalized_Final_Hidden_State`
    *   **Shape:** `(B, S, hidden_size)` -> `(1, 8, 2048)`

### 5. Language Model Head (LM Head)

*   **Input:** `Normalized_Final_Hidden_State`.
    *   **Shape:** `(B, S, hidden_size)` -> `(1, 8, 2048)`
*   **Process:** A final linear layer projects the hidden state for each token to the size of the vocabulary. This layer's weights are often tied to the input embedding weights for efficiency and better performance.
*   **Output:** `Logits`
    *   **Shape:** `(B, S, vocab_size)` -> `(1, 8, 32000)`

These `logits` represent the raw, unnormalized scores for each possible next token in the vocabulary for each position in the sequence. During text generation, typically only the logits for the *last* token (`(B, 1, vocab_size)`) are used to predict the next word. A softmax function would then convert these logits into probabilities.

## Autoregressive Text Generation Loop (Manual Implementation)

Instead of relying on `model.generate()`, we can implement the core autoregressive loop ourselves. This is crucial for understanding inference engineering, as it allows us to control each step and apply optimizations like the KV cache.

### The Algorithm: Step-by-Step

We'll use a **greedy decoding** strategy for simplicity, where at each step, the model predicts the token with the highest probability.

1.  **Initialize Input:** Start with your tokenized prompt (`input_ids`). This will be our initial sequence.
    *   `input_ids = tokenizer(prompt, return_tensors="pt").to(model.device)`

2.  **Initialize KV Cache:** For the first forward pass, there are no past key-value states, so we initialize `past_key_values = None`.

3.  **Define Stop Conditions:** Set a `max_new_tokens` limit or look for an End-Of-Sequence (EOS) token ID.

4.  **Generation Loop:** Repeat the following steps until a stop condition is met:

    a.  **Prepare Input for Model:**
        *   **First Iteration:** Pass the entire `input_ids` (the prompt) to the model.
        *   **Subsequent Iterations:** To optimize, we only pass the *most recently generated token* (`next_token`) as `input_ids`. The model will use the `past_key_values` from the previous step to efficiently compute attention for only the new token, avoiding recomputing for the entire past sequence. This is the **Key-Value (KV) Cache** in action.

    b.  **Model Forward Pass:** Call the model with the prepared input.
        *   `outputs = model(input_ids=current_input_ids, past_key_values=past_key_values)`

    c.  **Extract Logits:** From the model's `outputs`, get the `logits`. We are interested in the prediction for the *last* token in the current sequence.
        *   `logits = outputs.logits`
        *   `next_token_logits = logits[:, -1, :]` (Shape: `(B, vocab_size)`)

    d.  **Select Next Token (Greedy):** Choose the token ID with the highest probability from `next_token_logits`.
        *   `next_token = torch.argmax(next_token_logits, dim=-1)` (Shape: `(B,)`)

    e.  **Append to Generated Sequence:** Add this `next_token` to our running `input_ids` sequence.
        *   `input_ids = torch.cat([input_ids, next_token.unsqueeze(-1)], dim=-1)`

    f.  **Update KV Cache:** The model's output also includes updated `past_key_values` for the entire sequence processed up to this point. Store these for the next iteration.
        *   `past_key_values = outputs.past_key_values`

    g.  **Check Stop Condition:** If `next_token` is the EOS token ID or `max_new_tokens` have been generated, break the loop.

5.  **Decode Output:** Once the loop finishes, decode the final `input_ids` back into human-readable text.
    *   `generated_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)`

This loop efficiently generates text by only processing the newest token at each step, leveraging the KV cache to store attention states for previously generated tokens.

In [10]:
print("\n--- Manual Autoregressive Text Generation ---")

# 1. Define a prompt and tokenize it
prompt = "Tell me a short story about a brave knight."
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

# Generation parameters
max_new_tokens = 50

# Initialize KV cache
past_key_values = None

print(f"Initial prompt: '{prompt}'\n")

# Keep track of the generated tokens
# We'll append new tokens to `input_ids` in place for demonstration
# In a real-world scenario, you might have a separate list for generated tokens.

for i in range(max_new_tokens):
    # Prepare input for the current step
    # If it's the first step, pass the entire prompt.
    # Otherwise, pass only the last generated token.
    if past_key_values is None:
        # First iteration, process the full prompt
        current_input_ids = input_ids
    else:
        # Subsequent iterations, process only the last token
        current_input_ids = input_ids[:, -1].unsqueeze(-1) # Take last token and add batch dim

    # Model forward pass
    with torch.no_grad(): # Disable gradient calculation for inference
        outputs = model(
            input_ids=current_input_ids,
            past_key_values=past_key_values,
            use_cache=True # Ensure KV cache is used and returned
        )

    # Extract logits for the last token
    next_token_logits = outputs.logits[:, -1, :]

    # Select the next token (greedy decoding - argmax)
    next_token = torch.argmax(next_token_logits, dim=-1)

    # --- Inspection Prints ---
    print(f"\nIteration {i+1}:")
    print(f"  Current sequence length: {input_ids.shape[-1]} (after appending new token)")
    print(f"  Logits shape: {next_token_logits.shape}")
    print(f"  Selected token ID: {next_token.item()}")
    print(f"  Decoded token: '{tokenizer.decode(next_token)}'")
    print(f"  GPU device: {next_token.device}")
    # -------------------------

    # Append the new token to the sequence
    input_ids = torch.cat([input_ids, next_token.unsqueeze(-1)], dim=-1)

    # Update the KV cache for the next iteration
    past_key_values = outputs.past_key_values

    # Stop if EOS token is generated
    if next_token == tokenizer.eos_token_id:
        print("\n[EOS]")
        break

print("\n\n--- Generation Finished ---")
print("Full Generated Text:")
print(tokenizer.decode(input_ids[0], skip_special_tokens=True))



--- Manual Autoregressive Text Generation ---
Initial prompt: 'Tell me a short story about a brave knight.'


Iteration 1:
  Current sequence length: 12 (after appending new token)
  Logits shape: torch.Size([1, 32000])
  Selected token ID: 2
  Decoded token: '</s>'
  GPU device: cuda:0

[EOS]


--- Generation Finished ---
Full Generated Text:
Tell me a short story about a brave knight.


### Explanation: How Temperature Changes Logits/Probability Distribution

Temperature is a hyperparameter ($T$) that influences the randomness of a model's output. It works by scaling the logits *before* they are converted into a probability distribution using the `softmax` function. Mathematically, this looks like:

$$ P(token_i | context) = \text{softmax}(\frac{\text{logit}_i}{T}) $$

Let's break down the effect of $T$:

1.  **$T = 1.0$ (No Change):** When the temperature is 1, the logits are divided by 1, which means they remain unchanged. The `softmax` function then acts on the original logits, producing the model's 'raw' probability distribution.

2.  **$T < 1.0$ (Decreased Randomness, Sharper Distribution):** When the temperature is less than 1 (e.g., 0.7), dividing the logits by $T$ makes the larger (more positive) logits even larger and the smaller (more negative) logits even smaller *relative to each other*. This exaggerates the differences between the logit scores. When `softmax` is applied, the probability distribution becomes **sharper**, concentrating more probability mass on the tokens that originally had higher logits. This leads to less diverse, more predictable output, approaching greedy decoding as $T$ approaches 0.

    *Example: If logits are [2, 1, 0] and $T=0.5$, scaled logits become [4, 2, 0]. The differences are magnified, leading to a peakier distribution.*

3.  **$T > 1.0$ (Increased Randomness, Flatter Distribution):** When the temperature is greater than 1 (e.g., 1.5), dividing the logits by $T$ makes them closer to each other. The differences between the logit scores are reduced. When `softmax` is applied, the probability distribution becomes **flatter**, spreading the probability mass more evenly across a wider range of tokens. This encourages the model to pick lower-probability tokens, leading to more diverse, creative, but potentially less coherent output.

    *Example: If logits are [2, 1, 0] and $T=2.0$, scaled logits become [1, 0.5, 0]. The differences are compressed, leading to a flatter distribution.*


After scaling, instead of taking the `argmax` (which is greedy), we perform **multinomial sampling** from this temperature-modified probability distribution. This introduces the desired randomness, allowing tokens other than the absolute most probable one to be selected based on their scaled probabilities.

In [11]:
print("\n--- Manual Autoregressive Text Generation with Temperature Sampling ---")

# 1. Define a prompt and tokenize it
prompt = "Tell me a short story about a brave knight."
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

# Generation parameters
max_new_tokens = 50
temperature = 0.7 # New: Introduce temperature parameter

# Initialize KV cache
past_key_values = None

print(f"Initial prompt: '{prompt}'\n")
print(f"Temperature: {temperature}\n")

for i in range(max_new_tokens):
    # Prepare input for the current step
    if past_key_values is None:
        current_input_ids = input_ids
    else:
        current_input_ids = input_ids[:, -1].unsqueeze(-1)

    # Model forward pass
    with torch.no_grad():
        outputs = model(
            input_ids=current_input_ids,
            past_key_values=past_key_values,
            use_cache=True
        )

    # Extract logits for the last token
    next_token_logits = outputs.logits[:, -1, :]

    # NEW: Apply temperature scaling
    if temperature > 0:
        next_token_logits = next_token_logits / temperature

    # Convert logits to probabilities using softmax
    probs = torch.softmax(next_token_logits, dim=-1)

    # NEW: Sample next token from the probability distribution
    next_token = torch.multinomial(probs, num_samples=1)

    # --- Inspection Prints ---
    print(f"\nIteration {i+1}:")
    print(f"  Current sequence length: {input_ids.shape[-1]}")
    print(f"  Logits shape (pre-softmax): {next_token_logits.shape}") # Note: This is after temperature scaling
    print(f"  Selected token ID: {next_token.item()}")
    print(f"  Decoded token: '{tokenizer.decode(next_token)}'")
    print(f"  GPU device: {next_token.device}")
    # -------------------------

    # Append the new token to the sequence
    input_ids = torch.cat([input_ids, next_token], dim=-1)

    # Update the KV cache for the next iteration
    past_key_values = outputs.past_key_values

    # Stop if EOS token is generated
    if next_token == tokenizer.eos_token_id:
        print("\n[EOS]")
        break

print("\n\n--- Generation Finished ---")
print("Full Generated Text:")
print(tokenizer.decode(input_ids[0], skip_special_tokens=True))



--- Manual Autoregressive Text Generation with Temperature Sampling ---
Initial prompt: 'Tell me a short story about a brave knight.'

Temperature: 0.7


Iteration 1:
  Current sequence length: 12
  Logits shape (pre-softmax): torch.Size([1, 32000])
  Selected token ID: 29871
  Decoded token: '['']'
  GPU device: cuda:0

Iteration 2:
  Current sequence length: 13
  Logits shape (pre-softmax): torch.Size([1, 32000])
  Selected token ID: 29941
  Decoded token: '['3']'
  GPU device: cuda:0

Iteration 3:
  Current sequence length: 14
  Logits shape (pre-softmax): torch.Size([1, 32000])
  Selected token ID: 29889
  Decoded token: '['.']'
  GPU device: cuda:0

Iteration 4:
  Current sequence length: 15
  Logits shape (pre-softmax): torch.Size([1, 32000])
  Selected token ID: 7311
  Decoded token: '['Because']'
  GPU device: cuda:0

Iteration 5:
  Current sequence length: 16
  Logits shape (pre-softmax): torch.Size([1, 32000])
  Selected token ID: 306
  Decoded token: '['I']'
  GPU device: c

### Explanation: How Top-K Sampling Changes the Logits/Probability Distribution

Top-K sampling is a method to prune the vocabulary from which the next token is sampled. Instead of sampling from the entire vocabulary, we only consider the `k` tokens with the highest probabilities. This helps to prevent the model from generating nonsensical or low-probability tokens, especially when combined with temperature sampling.

Here's the process from an inference engineering perspective, focusing on tensor operations:

1.  **Start with Logits (after optional Temperature Scaling):** We begin with `next_token_logits` (shape `(1, vocab_size)`) which might have already been scaled by temperature.

2.  **Convert to Probabilities:** Apply `softmax` to these logits to get a probability distribution `probs` (shape `(1, vocab_size)`):
    $$ P(token_i | context) = \text{softmax}(\frac{\text{logit}_i}{T}) $$

3.  **Identify Top K Tokens:** We use `torch.topk(probs, k=top_k, dim=-1)`.
    *   This function returns two tensors: `top_k_values` (the actual probabilities of the top `k` tokens) and `top_k_indices` (the vocabulary IDs of those top `k` tokens).
    *   `top_k_values` shape: `(1, top_k)`
    *   `top_k_indices` shape: `(1, top_k)`

4.  **Create a Mask for Filtering:** To effectively limit the sampling to only these `top_k` tokens, we can create a tensor that has `-infinity` for all tokens *not* in the top `k`, and the original (or 0 for now) values for the top `k` tokens. This ensures that when `softmax` is applied again (implicitly in `multinomial` after setting values to 0 and re-normalizing), only the top `k` tokens have non-zero probability.
    *   A common way is to initialize a tensor `filter_value = -torch.inf` (or `float('-inf')`) and then set the `top_k_indices` logits back to their original values. For sampling from probabilities, we can just set non-top-k probabilities to 0.

5.  **Zero Out Non-Top-K Probabilities and Renormalize:**
    *   We create a `filtered_probs` tensor, initially all zeros, with the same shape as `probs` (`(1, vocab_size)`).
    *   Then, we use `filtered_probs.scatter_(-1, top_k_indices, top_k_values)` to place the `top_k_values` into `filtered_probs` at their `top_k_indices`. All other positions in `filtered_probs` remain zero.
    *   Finally, we re-normalize these `filtered_probs` by dividing them by their sum, ensuring they sum to 1 again. This is crucial because we removed a portion of the probability mass by discarding non-top-k tokens.

6.  **Sample from the Filtered Distribution:** Once `filtered_probs` is ready, we use `torch.multinomial(filtered_probs, num_samples=1)` to sample the `next_token` ID. This ensures that the sampled token will always be one of the `top_k` most probable tokens, while still introducing randomness based on their relative probabilities within that subset.

In [12]:
print("\n--- Manual Autoregressive Text Generation with Temperature & Top-K Sampling ---")

# 1. Define a prompt and tokenize it
prompt = "Tell me a short story about a brave knight."
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

# Generation parameters
max_new_tokens = 50
temperature = 0.7  # Temperature parameter
top_k = 50         # New: Top-K sampling parameter

# Initialize KV cache
past_key_values = None

print(f"Initial prompt: '{prompt}'\n")
print(f"Temperature: {temperature}")
print(f"Top-K: {top_k}\n")

for i in range(max_new_tokens):
    # Prepare input for the current step
    if past_key_values is None:
        current_input_ids = input_ids
    else:
        current_input_ids = input_ids[:, -1].unsqueeze(-1)

    # Model forward pass
    with torch.no_grad():
        outputs = model(
            input_ids=current_input_ids,
            past_key_values=past_key_values,
            use_cache=True
        )

    # Extract logits for the last token
    next_token_logits = outputs.logits[:, -1, :]

    # Apply temperature scaling
    if temperature > 0:
        next_token_logits = next_token_logits / temperature

    # Convert logits to probabilities using softmax
    probs = torch.softmax(next_token_logits, dim=-1)

    # NEW: Apply Top-K sampling
    if top_k > 0:
        # Get the top_k probabilities and their indices
        top_k_values, top_k_indices = torch.topk(probs, k=top_k, dim=-1)

        # Create a filtered probability distribution
        # Initialize a tensor of zeros with the same shape as probs
        filtered_probs = torch.zeros_like(probs, dtype=probs.dtype, device=probs.device)

        # Scatter the top_k_values back into the filtered_probs at their original indices
        filtered_probs.scatter_(-1, top_k_indices, top_k_values)

        # Renormalize the filtered probabilities so they sum to 1
        # Add a small epsilon to the sum to avoid division by zero if all top_k_values are 0
        filtered_probs = filtered_probs / filtered_probs.sum(dim=-1, keepdim=True)

        # Use the filtered_probs for sampling
        next_token = torch.multinomial(filtered_probs, num_samples=1)
    else:
        # If top_k is 0 or less, fall back to pure temperature sampling (or greedy if temperature=0)
        next_token = torch.multinomial(probs, num_samples=1)

    # --- Inspection Prints ---
    print(f"\nIteration {i+1}:")
    print(f"  Current sequence length: {input_ids.shape[-1]}")
    print(f"  Logits shape (pre-softmax/pre-topk): {next_token_logits.shape}")
    print(f"  Probabilities shape (after softmax): {probs.shape}")
    if top_k > 0:
      print(f"  Filtered probabilities sum: {filtered_probs.sum().item():.2f}")
      print(f"  Top-K values shape: {top_k_values.shape}")
    print(f"  Selected token ID: {next_token.item()}")
    print(f"  Decoded token: '{tokenizer.decode(next_token)}'")
    print(f"  GPU device: {next_token.device}")
    # -------------------------

    # Append the new token to the sequence
    input_ids = torch.cat([input_ids, next_token], dim=-1)

    # Update the KV cache for the next iteration
    past_key_values = outputs.past_key_values

    # Stop if EOS token is generated
    if next_token == tokenizer.eos_token_id:
        print("\n[EOS]")
        break

print("\n\n--- Generation Finished ---")
print("Full Generated Text:")
print(tokenizer.decode(input_ids[0], skip_special_tokens=True))



--- Manual Autoregressive Text Generation with Temperature & Top-K Sampling ---
Initial prompt: 'Tell me a short story about a brave knight.'

Temperature: 0.7
Top-K: 50


Iteration 1:
  Current sequence length: 12
  Logits shape (pre-softmax/pre-topk): torch.Size([1, 32000])
  Probabilities shape (after softmax): torch.Size([1, 32000])
  Filtered probabilities sum: 1.00
  Top-K values shape: torch.Size([1, 50])
  Selected token ID: 306
  Decoded token: '['I']'
  GPU device: cuda:0

Iteration 2:
  Current sequence length: 13
  Logits shape (pre-softmax/pre-topk): torch.Size([1, 32000])
  Probabilities shape (after softmax): torch.Size([1, 32000])
  Filtered probabilities sum: 1.00
  Top-K values shape: torch.Size([1, 50])
  Selected token ID: 505
  Decoded token: '['have']'
  GPU device: cuda:0

Iteration 3:
  Current sequence length: 14
  Logits shape (pre-softmax/pre-topk): torch.Size([1, 32000])
  Probabilities shape (after softmax): torch.Size([1, 32000])
  Filtered probabilities 

### Explanation: How Top-P (Nucleus) Sampling Changes the Logits/Probability Distribution

Top-p sampling, also known as nucleus sampling, is a dynamic and more flexible alternative to top-k sampling. Instead of fixing a number of tokens `k`, top-p sampling selects the smallest set of tokens whose cumulative probability mass exceeds a given threshold `p`. This means that the number of tokens considered for sampling can vary at each step, depending on the shape of the probability distribution.

Here's the process from an inference engineering perspective, focusing on tensor operations:

1.  **Start with Logits (after optional Temperature Scaling):** We begin with `next_token_logits` (shape `(1, vocab_size)`) which might have already been scaled by temperature.

2.  **Convert to Probabilities:** Apply `softmax` to these logits to get a probability distribution `probs` (shape `(1, vocab_size)`):
    $$ P(token_i | context) = \text{softmax}(\frac{\text{logit}_i}{T}) $$

3.  **Sort Probabilities:** Sort the probabilities in descending order. We need both the sorted probabilities and their original indices in the vocabulary.
    *   `sorted_probs, sorted_indices = torch.sort(probs, descending=True)`
    *   `sorted_probs` shape: `(1, vocab_size)`
    *   `sorted_indices` shape: `(1, vocab_size)`

4.  **Calculate Cumulative Probabilities:** Compute the cumulative sum of the `sorted_probs`.
    *   `cumulative_probs = torch.cumsum(sorted_probs, dim=-1)`
    *   `cumulative_probs` shape: `(1, vocab_size)`

5.  **Identify the Nucleus (Top-P Threshold):** Find the indices where the `cumulative_probs` first exceed the threshold `p`.
    *   `nucleus_mask = cumulative_probs < p` (This mask will be `True` for tokens within the nucleus and `False` outside)
    *   We need to include the first token that causes the sum to exceed `p`. So, we typically do `nucleus_mask[..., 1:] = cumulative_probs[..., :-1] < p` and `nucleus_mask[..., 0] = True` to ensure at least the most probable token is always included.
    *   A simpler, more direct approach often involves creating a mask of `True` for all tokens up to and including the one that pushes the cumulative sum over `p`.

6.  **Filter Probabilities:** Create a new probability distribution where all tokens *outside* the nucleus are set to zero (or negative infinity for logits before softmax). The tokens *inside* the nucleus retain their original probabilities.
    *   `filtered_probs = torch.zeros_like(probs, dtype=probs.dtype, device=probs.device)`
    *   `filtered_probs.scatter_(-1, sorted_indices[:, nucleus_mask[0]], sorted_probs[:, nucleus_mask[0]])` (This conceptually places the selected sorted probabilities back into their original vocabulary positions)
    *   A more common implementation applies `-torch.inf` to the logits directly for tokens outside the nucleus *before* the final softmax. For probabilities, we set them to zero.

7.  **Renormalize Filtered Probabilities:** The probabilities of the tokens within the nucleus must be re-normalized so that their sum equals 1. This ensures a valid probability distribution for sampling.
    *   `filtered_probs = filtered_probs / filtered_probs.sum(dim=-1, keepdim=True)`

8.  **Sample from the Filtered Distribution:** Finally, use `torch.multinomial(filtered_probs, num_samples=1)` to sample the `next_token` ID. This ensures that the sampled token is chosen from the selected nucleus, maintaining diversity while avoiding very low-probability tokens.

In [13]:
print("\n--- Manual Autoregressive Text Generation with Temperature & Top-P Sampling ---")

# 1. Define a prompt and tokenize it
prompt = "Tell me a short story about a brave knight."
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

# Generation parameters
max_new_tokens = 50
temperature = 0.7  # Temperature parameter
top_p = 0.9          # New: Top-P (nucleus) sampling parameter
# top_k = 50         # If you want to combine top-k, ensure it's applied before top-p, or choose one.
                     # For this example, we will use top-p instead of top-k.

# Initialize KV cache
past_key_values = None

print(f"Initial prompt: '{prompt}'\n")
print(f"Temperature: {temperature}")
print(f"Top-P: {top_p}\n")

for i in range(max_new_tokens):
    # Prepare input for the current step
    if past_key_values is None:
        current_input_ids = input_ids
    else:
        current_input_ids = input_ids[:, -1].unsqueeze(-1)

    # Model forward pass
    with torch.no_grad():
        outputs = model(
            input_ids=current_input_ids,
            past_key_values=past_key_values,
            use_cache=True
        )

    # Extract logits for the last token
    next_token_logits = outputs.logits[:, -1, :]

    # Apply temperature scaling
    if temperature > 0:
        next_token_logits = next_token_logits / temperature

    # Convert logits to probabilities using softmax
    probs = torch.softmax(next_token_logits, dim=-1)

    # NEW: Apply Top-P (nucleus) sampling
    if top_p > 0 and top_p < 1.0:
        # Sort probabilities in descending order
        sorted_probs, sorted_indices = torch.sort(probs, descending=True)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

        # Create a mask to identify the nucleus
        # The `nucleus_mask` will be True for tokens to keep.
        # Add a True at the beginning to always include at least one token.
        nucleus_mask = cumulative_probs - sorted_probs > top_p
        sorted_probs[nucleus_mask] = 0.0 # Set probabilities outside nucleus to 0

        # Renormalize the filtered probabilities
        filtered_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)

        # Sample from the filtered distribution
        # We need to map the sampled index back to the original vocabulary index
        sampled_index_in_sorted = torch.multinomial(filtered_probs, num_samples=1)
        next_token = sorted_indices.gather(-1, sampled_index_in_sorted)

    # Fallback if top_p is not applied or invalid (e.g., pure temperature sampling)
    else:
        next_token = torch.multinomial(probs, num_samples=1)

    # --- Inspection Prints ---
    print(f"\nIteration {i+1}:")
    print(f"  Current sequence length: {input_ids.shape[-1]}")
    print(f"  Logits shape (pre-softmax/pre-sampling): {next_token_logits.shape}")
    print(f"  Probabilities shape (after softmax): {probs.shape}")
    if top_p > 0 and top_p < 1.0:
        print(f"  Sorted probabilities (first 10): {sorted_probs[0, :10].tolist()}")
        print(f"  Cumulative probabilities (first 10): {cumulative_probs[0, :10].tolist()}")
        print(f"  Number of tokens in nucleus: {torch.sum(filtered_probs > 0).item()}")
        print(f"  Filtered probabilities sum: {filtered_probs.sum().item():.2f}")
    print(f"  Selected token ID: {next_token.item()}")
    print(f"  Decoded token: '{tokenizer.decode(next_token)}'")
    print(f"  GPU device: {next_token.device}")
    # -------------------------

    # Append the new token to the sequence
    input_ids = torch.cat([input_ids, next_token], dim=-1)

    # Update the KV cache for the next iteration
    past_key_values = outputs.past_key_values

    # Stop if EOS token is generated
    if next_token == tokenizer.eos_token_id:
        print("\n[EOS]")
        break

print("\n\n--- Generation Finished ---")
print("Full Generated Text:")
print(tokenizer.decode(input_ids[0], skip_special_tokens=True))



--- Manual Autoregressive Text Generation with Temperature & Top-P Sampling ---
Initial prompt: 'Tell me a short story about a brave knight.'

Temperature: 0.7
Top-P: 0.9


Iteration 1:
  Current sequence length: 12
  Logits shape (pre-softmax/pre-sampling): torch.Size([1, 32000])
  Probabilities shape (after softmax): torch.Size([1, 32000])
  Sorted probabilities (first 10): [0.28515625, 0.1435546875, 0.119140625, 0.0869140625, 0.0771484375, 0.052978515625, 0.0250244140625, 0.018310546875, 0.01177978515625, 0.0111083984375]
  Cumulative probabilities (first 10): [0.28515625, 0.4296875, 0.546875, 0.6328125, 0.7109375, 0.765625, 0.7890625, 0.80859375, 0.8203125, 0.83203125]
  Number of tokens in nucleus: 19
  Filtered probabilities sum: 1.00
  Selected token ID: 2
  Decoded token: '['</s>']'
  GPU device: cuda:0

[EOS]


--- Generation Finished ---
Full Generated Text:
Tell me a short story about a brave knight.


In [15]:
def generate_text_with_sampling_benchmark(
    model,
    tokenizer,
    prompt,
    max_new_tokens=50,
    temperature=0.7,
    top_p=0.9
):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    input_token_count = input_ids.shape[-1]
    past_key_values = None
    generated_token_ids = []

    # CUDA Events for accurate timing
    start_event = torch.cuda.Event(enable_timing=True)
    first_token_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    ttft_ms = 0.0

    start_event.record() # Record start of generation

    for i in range(max_new_tokens):
        # Prepare input for the current step
        if past_key_values is None:
            current_input_ids = input_ids
        else:
            current_input_ids = generated_token_ids[-1].unsqueeze(-1) # Use only the last generated token

        # Model forward pass
        with torch.no_grad():
            outputs = model(
                input_ids=current_input_ids,
                past_key_values=past_key_values,
                use_cache=True
            )

        # Extract logits for the last token
        next_token_logits = outputs.logits[:, -1, :]

        # Apply temperature scaling
        if temperature > 0:
            next_token_logits = next_token_logits / temperature

        # Convert logits to probabilities using softmax
        probs = torch.softmax(next_token_logits, dim=-1)

        # Apply Top-P (nucleus) sampling
        if top_p > 0 and top_p < 1.0:
            sorted_probs, sorted_indices = torch.sort(probs, descending=True)
            cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

            # Create a mask to identify the nucleus
            # We want to select the smallest set of tokens whose cumulative probability exceeds top_p
            # This masks out tokens where `cumulative_probs` is GREATER than top_p,
            # ensuring that at least one token is always included even if its cumulative_probs is > top_p
            # by making sure `cumulative_probs - sorted_probs` is used before comparison.
            # Add a True at the beginning to always include at least one token. This ensures `cumulative_probs - sorted_probs > top_p` does not exclude the first item incorrectly.
            # Simplified mask: Find where cumulative sum is less than top_p, and set the first one that exceeds to be included as well.
            mask_values = cumulative_probs - sorted_probs
            nucleus_mask = mask_values < top_p

            sorted_probs[~nucleus_mask] = 0.0 # Set probabilities outside nucleus to 0

            # Renormalize the filtered probabilities
            filtered_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)

            # Sample from the filtered distribution
            sampled_index_in_sorted = torch.multinomial(filtered_probs, num_samples=1)
            next_token = sorted_indices.gather(-1, sampled_index_in_sorted)
        else:
            next_token = torch.multinomial(probs, num_samples=1)

        # Record TTFT after the first token is generated and sampled
        if i == 0:
            first_token_event.record()
            torch.cuda.synchronize() # Ensure first_token_event is recorded accurately
            ttft_ms = start_event.elapsed_time(first_token_event)

        generated_token_ids.append(next_token)
        # Stop if EOS token is generated
        if next_token == tokenizer.eos_token_id:
            break

    end_event.record() # Record end of generation
    torch.cuda.synchronize() # Ensure end_event is recorded accurately

    total_latency_ms = start_event.elapsed_time(end_event)
    output_token_count = len(generated_token_ids)

    return input_token_count, output_token_count, ttft_ms, total_latency_ms

In [16]:
print("\n--- Benchmarking Custom LLM Inference Function ---")

# Benchmarking parameters
benchmark_prompt = "Tell me a short story about a brave knight in a magical forest facing a dragon."
num_warmup_runs = 5
num_measurement_runs = 10
max_new_tokens = 100
temperature = 0.7
top_p = 0.9

print(f"Prompt: '{benchmark_prompt}'")
print(f"Max new tokens: {max_new_tokens}")
print(f"Temperature: {temperature}")
print(f"Top-P: {top_p}")
print(f"Warm-up runs: {num_warmup_runs}")
print(f"Measurement runs: {num_measurement_runs}\n")

# Store results
all_input_tokens = []
all_output_tokens = []
all_ttft_ms = []
all_total_latency_ms = []
all_tpot_ms_per_token = []
all_output_tps = []

# --- Warm-up Runs ---
print("Starting warm-up runs...")
for i in range(num_warmup_runs):
    _, _, _, _ = generate_text_with_sampling_benchmark(
        model, tokenizer, benchmark_prompt, max_new_tokens, temperature, top_p
    )
    print(f"Warm-up run {i+1}/{num_warmup_runs} complete.")
print("Warm-up complete.\n")

# --- Measurement Runs ---
print("Starting measurement runs...")
for i in range(num_measurement_runs):
    input_tokens, output_tokens, ttft_ms, total_latency_ms = generate_text_with_sampling_benchmark(
        model, tokenizer, benchmark_prompt, max_new_tokens, temperature, top_p
    )

    all_input_tokens.append(input_tokens)
    all_output_tokens.append(output_tokens)
    all_ttft_ms.append(ttft_ms)
    all_total_latency_ms.append(total_latency_ms)

    # Calculate TPOT and TPS for this run
    if output_tokens > 1:
        tpot_ms_per_token = (total_latency_ms - ttft_ms) / (output_tokens - 1)
    else: # If only one token generated, TTFT is total latency, TPOT is not meaningful
        tpot_ms_per_token = total_latency_ms # Or 0.0 or float('nan') based on preference

    output_tps = (output_tokens / (total_latency_ms / 1000)) if total_latency_ms > 0 else 0.0

    all_tpot_ms_per_token.append(tpot_ms_per_token)
    all_output_tps.append(output_tps)

    print(f"Measurement run {i+1}/{num_measurement_runs} complete. Output tokens: {output_tokens}, TTFT: {ttft_ms:.2f}ms, Total Latency: {total_latency_ms:.2f}ms")

print("\n--- Benchmark Results (Averages) ---")

avg_input_tokens = sum(all_input_tokens) / len(all_input_tokens)
avg_output_tokens = sum(all_output_tokens) / len(all_output_tokens)
avg_ttft_ms = sum(all_ttft_ms) / len(all_ttft_ms)
avg_total_latency_ms = sum(all_total_latency_ms) / len(all_total_latency_ms)
avg_tpot_ms_per_token = sum(all_tpot_ms_per_token) / len(all_tpot_ms_per_token)
avg_output_tps = sum(all_output_tps) / len(all_output_tps)

print(f"Average Input Tokens: {avg_input_tokens:.1f}")
print(f"Average Output Tokens: {avg_output_tokens:.1f}")
print(f"Average Time To First Token (TTFT): {avg_ttft_ms:.2f} ms")
print(f"Average Total Generation Latency: {avg_total_latency_ms:.2f} ms")
print(f"Average Time Per Output Token (TPOT): {avg_tpot_ms_per_token:.2f} ms/token")
print(f"Average Output Tokens Per Second (TPS): {avg_output_tps:.2f} tokens/s")

# Optional: Calculate percentiles for latency metrics
import numpy as np
p50_ttft = np.percentile(all_ttft_ms, 50)
p95_ttft = np.percentile(all_ttft_ms, 95)
p99_ttft = np.percentile(all_ttft_ms, 99)

p50_total_latency = np.percentile(all_total_latency_ms, 50)
p95_total_latency = np.percentile(all_total_latency_ms, 95)
p99_total_latency = np.percentile(all_total_latency_ms, 99)

print("\n--- Latency Percentiles ---")
print(f"TTFT p50: {p50_ttft:.2f} ms, p95: {p95_ttft:.2f} ms, p99: {p99_ttft:.2f} ms")
print(f"Total Latency p50: {p50_total_latency:.2f} ms, p95: {p95_total_latency:.2f} ms, p99: {p99_total_latency:.2f} ms")



--- Benchmarking Custom LLM Inference Function ---
Prompt: 'Tell me a short story about a brave knight in a magical forest facing a dragon.'
Max new tokens: 100
Temperature: 0.7
Top-P: 0.9
Warm-up runs: 5
Measurement runs: 10

Starting warm-up runs...
Warm-up run 1/5 complete.
Warm-up run 2/5 complete.
Warm-up run 3/5 complete.
Warm-up run 4/5 complete.
Warm-up run 5/5 complete.
Warm-up complete.

Starting measurement runs...
Measurement run 1/10 complete. Output tokens: 2, TTFT: 76.95ms, Total Latency: 152.61ms
Measurement run 2/10 complete. Output tokens: 2, TTFT: 85.15ms, Total Latency: 193.63ms
Measurement run 3/10 complete. Output tokens: 2, TTFT: 74.84ms, Total Latency: 165.88ms
Measurement run 4/10 complete. Output tokens: 6, TTFT: 74.40ms, Total Latency: 523.42ms
Measurement run 5/10 complete. Output tokens: 4, TTFT: 75.69ms, Total Latency: 324.27ms
Measurement run 6/10 complete. Output tokens: 1, TTFT: 77.35ms, Total Latency: 77.75ms
Measurement run 7/10 complete. Output toke

### Experiment Design: Impact of Output Length on Inference Latency

To understand how the length of the generated output affects performance metrics, we will run our benchmark function for different `max_new_tokens` values. We will keep the prompt and sampling parameters (`temperature`, `top_p`) constant.

**Hypothesized Behavior of Metrics with Increasing Output Length:**

1.  **Time To First Token (TTFT):**
    *   **Expectation:** Should remain relatively **constant**. TTFT primarily depends on the processing of the input prompt (context encoding) and the first forward pass of the model. It is largely independent of how many tokens are generated *after* the first one.

2.  **Time Per Output Token (TPOT):**
    *   **Expectation:** Should also remain relatively **constant** after the initial setup. TPOT measures the average time for each subsequent token. Each additional token requires a single forward pass with the KV cache, which is a fairly consistent operation. For very short generations, the overhead might slightly skew the average, but it should stabilize as generation length increases.

3.  **Total Generation Latency:**
    *   **Expectation:** Should increase **linearly** with the number of output tokens. This is because the total latency is essentially `TTFT + (Number of Output Tokens - 1) * TPOT`. As `Number of Output Tokens` increases, the `(N-1) * TPOT` component dominates, leading to a linear increase.

4.  **Output Tokens Per Second (Output TPS):**
    *   **Expectation:** Should **increase and then plateau**. For very short generations (e.g., 1-10 tokens), the TTFT overhead represents a significant portion of the total latency, leading to a lower TPS. As the number of output tokens grows, the fixed TTFT overhead becomes proportionally smaller, and the TPS approaches `1000 / TPOT` (its theoretical maximum based on sequential token generation). It might even slightly decrease for extremely long sequences if memory management or other system-level overheads become significant.

**Experiment Parameters:**

*   **Output Lengths (`max_new_tokens`):** We'll test 10, 50, 100, and 500 tokens.
*   **Prompt, Temperature, Top-P:** Will remain consistent across all runs.
*   **Warm-up & Measurement Runs:** Same as before, to ensure stable and reliable averages.

In [ ]:
print("\n--- Experiment: Output Length vs. Latency Metrics ---")

# Experiment parameters
experiment_prompt = "Write a detailed story about a wizard's journey through a mystical land to find a lost artifact. Include descriptions of the challenges faced and the creatures encountered."
output_lengths_to_test = [10, 50, 100, 500] # Number of new tokens to generate
num_warmup_runs = 3 # Reduced warm-up for quicker demonstration
num_measurement_runs = 5 # Reduced measurement runs for quicker demonstration
temperature = 0.7
top_p = 0.9

print(f"Experiment Prompt: '{experiment_prompt[:80]}...' (truncated for display)")
print(f"Output lengths to test (max_new_tokens): {output_lengths_to_test}")
print(f"Temperature: {temperature}, Top-P: {top_p}")
print(f"Warm-up runs per length: {num_warmup_runs}")
print(f"Measurement runs per length: {num_measurement_runs}\n")

all_experiment_results = {}

for current_max_new_tokens in output_lengths_to_test:
    print(f"\n--- Running benchmark for max_new_tokens = {current_max_new_tokens} ---")

    # Store results for current output length
    current_run_input_tokens = []
    current_run_output_tokens = []
    current_run_ttft_ms = []
    current_run_total_latency_ms = []
    current_run_tpot_ms_per_token = []
    current_run_output_tps = []

    # Warm-up Runs
    print(f"Starting warm-up runs for {current_max_new_tokens} tokens...")
    for _ in range(num_warmup_runs):
        _, _, _, _ = generate_text_with_sampling_benchmark(
            model, tokenizer, experiment_prompt, current_max_new_tokens, temperature, top_p
        )
    print("Warm-up complete.")

    # Measurement Runs
    print(f"Starting measurement runs for {current_max_new_tokens} tokens...")
    for i in range(num_measurement_runs):
        input_tokens, output_tokens, ttft_ms, total_latency_ms = generate_text_with_sampling_benchmark(
            model, tokenizer, experiment_prompt, current_max_new_tokens, temperature, top_p
        )

        current_run_input_tokens.append(input_tokens)
        current_run_output_tokens.append(output_tokens)
        current_run_ttft_ms.append(ttft_ms)
        current_run_total_latency_ms.append(total_latency_ms)

        if output_tokens > 1:
            tpot_ms_per_token = (total_latency_ms - ttft_ms) / (output_tokens - 1)
        else:
            tpot_ms_per_token = ttft_ms # If only one token, TTFT is essentially TPOT

        output_tps = (output_tokens / (total_latency_ms / 1000)) if total_latency_ms > 0 else 0.0

        current_run_tpot_ms_per_token.append(tpot_ms_per_token)
        current_run_output_tps.append(output_tps)
        print(f"  Run {i+1}/{num_measurement_runs}: Output tokens: {output_tokens}, TTFT: {ttft_ms:.2f}ms, Total Latency: {total_latency_ms:.2f}ms")

    # Calculate averages for this output length
    avg_ttft_ms = sum(current_run_ttft_ms) / len(current_run_ttft_ms)
    avg_total_latency_ms = sum(current_run_total_latency_ms) / len(current_run_total_latency_ms)
    avg_tpot_ms_per_token = sum(current_run_tpot_ms_per_token) / len(current_run_tpot_ms_per_token)
    avg_output_tps = sum(current_run_output_tps) / len(current_run_output_tps)
    avg_output_tokens = sum(current_run_output_tokens) / len(current_run_output_tokens)

    all_experiment_results[current_max_new_tokens] = {
        "avg_output_tokens": avg_output_tokens,
        "avg_ttft_ms": avg_ttft_ms,
        "avg_total_latency_ms": avg_total_latency_ms,
        "avg_tpot_ms_per_token": avg_tpot_ms_per_token,
        "avg_output_tps": avg_output_tps
    }

print("\n--- Experiment Summary Results ---")
print("| Max Output Tokens | Avg Output Tokens | Avg TTFT (ms) | Avg Total Latency (ms) | Avg TPOT (ms/token) | Avg Output TPS (tokens/s) |")
print("|-------------------|-------------------|---------------|------------------------|---------------------|---------------------------|")
for max_tokens, results in all_experiment_results.items():
    print(f"| {max_tokens:<17} | {results['avg_output_tokens']:<17.1f} | {results['avg_ttft_ms']:<13.2f} | {results['avg_total_latency_ms']:<22.2f} | {results['avg_tpot_ms_per_token']:<19.2f} | {results['avg_output_tps']:<25.2f} |")



--- Experiment: Output Length vs. Latency Metrics ---
Experiment Prompt: 'Write a detailed story about a wizard's journey through a mystical land to find ...' (truncated for display)
Output lengths to test (max_new_tokens): [10, 50, 100, 500]
Temperature: 0.7, Top-P: 0.9
Warm-up runs per length: 3
Measurement runs per length: 5


--- Running benchmark for max_new_tokens = 10 ---
Starting warm-up runs for 10 tokens...
Warm-up complete.
Starting measurement runs for 10 tokens...
  Run 1/5: Output tokens: 10, TTFT: 80.37ms, Total Latency: 793.47ms
  Run 2/5: Output tokens: 10, TTFT: 78.50ms, Total Latency: 792.90ms
  Run 3/5: Output tokens: 10, TTFT: 78.96ms, Total Latency: 796.15ms
  Run 4/5: Output tokens: 10, TTFT: 79.04ms, Total Latency: 798.00ms
  Run 5/5: Output tokens: 10, TTFT: 80.63ms, Total Latency: 809.45ms

--- Running benchmark for max_new_tokens = 50 ---
Starting warm-up runs for 50 tokens...
Warm-up complete.
Starting measurement runs for 50 tokens...
  Run 1/5: Output tok

### Benchmarking GPU Inference Latency in PyTorch: Why Asynchronous Execution Matters

When working with GPUs in PyTorch (and other frameworks), operations are inherently **asynchronous**. This means that when you call a PyTorch function that uses the GPU (e.g., `model(input_ids)` or `torch.matmul()`), the Python code *does not wait* for the GPU computation to complete before moving to the next line. Instead, the operation is queued on the GPU's stream, and Python immediately continues execution.

#### Why `time.time()` is Misleading:

If you try to measure the duration of a GPU operation using `time.time()` like this:

```python
import time
start_time = time.time()
gpu_operation()
end_time = time.time()
duration = end_time - start_time
```

The `duration` you measure will likely be very small, often close to zero, or much smaller than the actual GPU computation time. This is because `time.time()` measures the time the Python interpreter spends on those lines, which primarily involves *queueing the operation* onto the GPU. The actual execution of the operation on the GPU happens in parallel and often finishes much later.

This can lead to:
*   **Underestimation of GPU workload:** Your timings will not reflect the true compute cost.
*   **Difficulty in debugging:** You might incorrectly assume an operation is fast when it's actually a bottleneck.
*   **Incorrect performance comparisons:** Benchmarking different models or optimizations will yield inaccurate results.

### Correct Benchmarking with CUDA Synchronization and Events

To accurately measure GPU execution time, you need to explicitly tell PyTorch (and CUDA) to wait until all queued operations are complete. There are two primary ways to do this:

1.  **`torch.cuda.synchronize()` (Simple, but can obscure true parallelism):**
    *   This function blocks the CPU until all computations on the *default CUDA stream* on the *current device* have finished. While simple to use, it can artificially inflate timings if you have multiple streams or overlapping CPU/GPU work, as it forces everything to halt.

2.  **`torch.cuda.Event` (Recommended for precision and advanced use cases):**
    *   CUDA events are lightweight markers that can be placed in a CUDA stream. They allow you to record specific points in time on the GPU. You can then measure the time elapsed between two events. Crucially, `event.synchronize()` only waits for that specific event to be recorded, making it more flexible for measuring specific sections of GPU code without necessarily blocking the entire stream prematurely. They are essential for profiling and understanding complex asynchronous workflows.
    *   To use them, you `record` an event, perform your GPU operations, `record` another event, then `synchronize` the second event, and finally `elapsed_time()` between them.

In [14]:
import torch
import time

# Ensure we're on a CUDA device for this demonstration
if not torch.cuda.is_available():
    print("CUDA not available. Skipping GPU benchmarking demonstration.")
    exit()

device = torch.device("cuda")
print(f"Benchmarking on device: {device}")

# Prepare some dummy data for a simple GPU operation
input_size = 4096 # Large enough to see meaningful GPU time
matrix_a = torch.randn(input_size, input_size, device=device, dtype=torch.float16)
matrix_b = torch.randn(input_size, input_size, device=device, dtype=torch.float16)

print(f"Performing a matrix multiplication on GPU ({input_size}x{input_size} @ {input_size}x{input_size})...")

### Method 1: Misleading CPU-based `time.time()`

start_cpu_time = time.time()
result_cpu_time = torch.matmul(matrix_a, matrix_b) # GPU operation queued
end_cpu_time = time.time()
misleading_duration = (end_cpu_time - start_cpu_time) * 1000
print(f"\n1. Misleading CPU-based timing (time.time()): {misleading_duration:.3f} ms")

### Method 2: Using `torch.cuda.synchronize()`

# Warm-up run to ensure GPU is ready
for _ in range(5):
    _ = torch.matmul(matrix_a, matrix_b)
torch.cuda.synchronize() # Wait for warm-up to finish

start_sync_time = time.time()
result_sync = torch.matmul(matrix_a, matrix_b) # GPU operation queued
torch.cuda.synchronize() # BLOCK CPU until GPU operation is complete
end_sync_time = time.time()
sync_duration = (end_sync_time - start_sync_time) * 1000
print(f"2. Accurate timing with torch.cuda.synchronize(): {sync_duration:.3f} ms")

### Method 3: Using `torch.cuda.Event` (Recommended for precision)

# Warm-up run
for _ in range(5):
    _ = torch.matmul(matrix_a, matrix_b)

# Create CUDA events
start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)

# Record start event before the GPU operation
start_event.record()

result_event = torch.matmul(matrix_a, matrix_b) # GPU operation queued

# Record end event after the GPU operation
end_event.record()

# Wait for the end event to complete (synchronizes implicitly)
end_event.synchronize()

# Calculate elapsed time between events in milliseconds
event_duration = start_event.elapsed_time(end_event)
print(f"3. Accurate timing with torch.cuda.Event: {event_duration:.3f} ms")

# Verify results (optional)
# assert torch.equal(result_cpu_time, result_sync)
# assert torch.equal(result_cpu_time, result_event)

print("\nAs you can see, the CPU-based timing is significantly lower because it doesn't wait for the GPU computation to complete. `torch.cuda.synchronize()` and `torch.cuda.Event` provide much more accurate measurements of the actual GPU workload.")


Benchmarking on device: cuda
Performing a matrix multiplication on GPU (4096x4096 @ 4096x4096)...

1. Misleading CPU-based timing (time.time()): 144.800 ms
2. Accurate timing with torch.cuda.synchronize(): 7.282 ms
3. Accurate timing with torch.cuda.Event: 6.544 ms

As you can see, the CPU-based timing is significantly lower because it doesn't wait for the GPU computation to complete. `torch.cuda.synchronize()` and `torch.cuda.Event` provide much more accurate measurements of the actual GPU workload.
